In [2]:
import pandas as pd
import altair as alt

In [3]:

# load a sample dataset as a pandas DataFrame
from altair.datasets import data
cars = data.cars()

# make the chart
alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
).interactive()

alt.Chart(...)

In [16]:
tesis = pd.read_csv("tesis_joined_data.csv", dtype=str, index_col=0)
sentencias = pd.read_csv("sentencias_joined_data.csv", dtype=str, index_col=0)

In [17]:
tesis.columns

Index(['huellaDigital', 'idTesis', 'epoca', 'localizacion', 'anio', 'mes',
       'instancia', 'organoJuris', 'fuente', 'materias', 'tipoTesis', 'tesis',
       'rubro', 'texto', 'precedentes', 'notaPublica', 'anexos',
       'fuenteExtraccion', 'ministro', 'votacion', 'main_materia'],
      dtype='str')

In [18]:
sentencias.columns

Index(['expediente', 'pertenencia', 'ministro', 'tema',
       'organoJurisdiccionalOrigen', 'expedienteOrigen', 'organoResolvio',
       'fechaResolucion', 'resolucion', 'ponente', 'documentoSentencia',
       'votacion', 'asuntosAcumulados', 'tipoAsunto', 'huellaDigital',
       'fuenteExtraccion', 'idSentencia', 'urlInternet', 'cleanDate', 'anio',
       'mes', 'votos'],
      dtype='str')

In [36]:
counts_tesis = tesis.groupby('anio')['idTesis'].count().to_frame().reset_index()
counts_tesis


,anio,idTesis
0,2015,862
1,2016,759
2,2017,771
3,2018,731
4,2019,478
5,2020,272
6,2021,251
7,2022,336
8,2023,359
9,2024,360


In [37]:
counts_sentencias = sentencias.groupby('anio')['expediente'].count().to_frame().reset_index()
counts_sentencias

,anio,expediente
0,2015,829
1,2016,6085
2,2017,7247
3,2018,2699
4,2019,6646
5,2020,3717
6,2021,3100
7,2022,2569
8,2023,2726
9,2024,3791


In [40]:
chart_tesis = alt.Chart(counts_tesis).mark_line().encode(
    x='anio',
    y='idTesis'
).properties(
    title='Tesis emitidas'
)

chart_sentencias = alt.Chart(counts_sentencias).mark_line().encode(
    x='anio',
    y='expediente'
)



chart_sentencias

alt.Chart(...)

In [47]:
materias = tesis.groupby(['anio', 'main_materia'])['idTesis'].count().to_frame().reset_index()
materias

,anio,main_materia,idTesis
0,2015,Administrativa,89
1,2015,Civil,48
2,2015,Común,192
3,2015,Constitucional,501
4,2015,Laboral,14
...,...,...,...
65,2025,Penal,80
66,2026,Civil,2
67,2026,Común,1
68,2026,Constitucional,4


In [70]:
chart = (
    alt.Chart(materias)
    .transform_window(
        rank='rank()',
        sort=[alt.SortField('idTesis', order='descending')],
        groupby=['anio']
    )
    .mark_line(point=True)
    .encode(
        x=alt.X('anio:O', title='Year'),
        y=alt.Y('rank:O', title='Rank'),
        color='main_materia:N',
        tooltip=['anio:N','main_materia:N','idTesis:Q','rank:Q']
    )
    .properties(width=500, height=500)
)

chart

alt.Chart(...)

In [53]:
chart_2 = (
    alt.Chart(materias)
    .transform_window(
        rank='rank()',
        sort=[alt.SortField('idTesis', order='descending')],
        groupby=['anio']
    )
    .transform_filter('datum.rank <= 10')
    .mark_bar()
    .encode(
        x='anio:O',
        y='idTesis:Q',
        color='main_materia:N',
        column='main_materia:N'
    )
)

chart_2

alt.Chart(...)

In [43]:

chart = ( 
        alt.Chart(materias)
        .mark_rect()
        .encode(
            x=alt.X("main_materia:N", title="Materia"),
            y=alt.Y("anio:N", title="Anio"),
            color=alt.Color(
                "Count:idTesis", title="Total tesis", scale=alt.Scale(scheme="blues")

            ),
            tooltip=["anio:N", "main_materia:N", "Count:idTesis"],
        )
        .properties(title="Distribution of AQI categories in")
    )
chart 

ValueError: "idTesis" is not one of the valid encoding data types: O, N, Q, T, G.
For more details, see https://altair-viz.github.io/user_guide/encodings/index.html#encoding-data-types. If you are trying to use a column name that contains a colon, prefix it with a backslash; for example "column\:name" instead of "column:name".

alt.Chart(...)

In [74]:
sentencias_ministro = sentencias.groupby(['anio', 'ministro'])['expediente'].count().to_frame().reset_index()
sentencias_ministro

,anio,ministro,expediente
0,2015,ALBERTO PÉREZ DAYÁN,102
1,2015,ALFREDO GUTIÉRREZ ORTIZ MENA,81
2,2015,ARTURO ZALDÍVAR LELO DE LARREA,84
3,2015,JORGE MARIO PARDO REBOLLEDO,90
4,2015,JOSÉ FERNANDO FRANCO GONZÁLEZ SALAS,109
...,...,...,...
132,2026,MARGARITA BEATRIZ LUNA RAMOS,15
133,2026,MARÍA ESTELA RÍOS GONZÁLEZ,3
134,2026,OLGA MARÍA DEL CARMEN SÁNCHEZ CORDERO,8
135,2026,SARA IRENE HERRERÍAS GUERRA,6


In [78]:
highlight = alt.selection_point(fields=['ministro'], on='mouseover', empty=False)

chart = (
    alt.Chart(sentencias_ministro)
    .mark_line()
    .encode(
        x='anio:O',
        y='expediente:Q',
        color='ministro:N',
        opacity=alt.condition(highlight, alt.value(1), alt.value(0.2)),
        tooltip=['anio','ministro','expediente']
    )
    .add_params(highlight)
    .properties(width=700, height=400)
)

chart

alt.Chart(...)

In [80]:
chart = (
    alt.Chart(sentencias_ministro)
    .transform_window(
        cumulative='sum(expediente)',
        sort=[alt.SortField('anio')],
        groupby=['ministro']
    )
    .mark_line()
    .encode(
        x='anio:O',
        y='cumulative:Q',
        color='ministro:N'
    )
)

chart

alt.Chart(...)

In [81]:

df = sentencias_ministro.copy()
df["anio"] = df["anio"].astype(int)
df = df.sort_values(["ministro", "anio"])

# 1) Cumulative per minister
df["cumulative"] = df.groupby("ministro")["expediente"].cumsum()

# 2) Average cumulative across ministers, per year
avg_cum = (
    df.groupby("anio", as_index=False)["cumulative"]
      .mean()
      .rename(columns={"cumulative": "avg_cumulative"})
)

# 3) Merge and compute difference vs average
df = df.merge(avg_cum, on="anio", how="left")
df["diff_vs_avg"] = df["cumulative"] - df["avg_cumulative"]

# Dropdown to pick one minister
minister_options = sorted(df["ministro"].unique())
dropdown = alt.binding_select(options=minister_options, name="Minister: ")
sel = alt.selection_point(fields=["ministro"], bind=dropdown, value=minister_options[0])

base = (
    alt.Chart(df)
    .add_params(sel)
    .transform_filter(sel)
)

# Zero baseline
zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule().encode(y="y:Q")

chart = (
    (zero + base.mark_line(point=True))
    .encode(
        x=alt.X("anio:O", title="Year"),
        y=alt.Y("diff_vs_avg:Q", title="Cumulative rulings vs average (difference)"),
        tooltip=[
            alt.Tooltip("anio:O", title="Year"),
            alt.Tooltip("ministro:N", title="Minister"),
            alt.Tooltip("expediente:Q", title="Rulings (year)"),
            alt.Tooltip("cumulative:Q", title="Cumulative"),
            alt.Tooltip("avg_cumulative:Q", title="Avg cumulative"),
            alt.Tooltip("diff_vs_avg:Q", title="Diff vs avg"),
        ],
    )
    .properties(width=750, height=420, title="Selected minister vs average (cumulative difference)")
)

chart

alt.LayerChart(...)

In [83]:
chart = (
    alt.Chart(sentencias_ministro)
    .mark_rect()
    .encode(
        x='anio:O',
        y=alt.Y('ministro:N', sort='-x'),
        color=alt.Color('expediente:Q', title='Rulings')
    )
    .properties(width=800, height=500)
)
chart

alt.Chart(...)

In [87]:
heatmap_data = (
    tesis
    .groupby(["anio", "ministro", "main_materia"])
    .size()
    .reset_index(name="counts")
)

In [ ]:


chart

alt.Chart(...)